# **Encoder-Decoder (Seq2Seq)**

Paper: [Learning Phrase Representations using RNN Encoder-Decoder for Statistical Machine Translation](https://emnlp2014.org/papers/pdf/EMNLP2014179.pdf)

In the [previous notebook](../7_Recurrent_Neural_Networks/rnn.ipynb) we saw that a plain RNN takes a sequence and, at every timestep, produces an output. This works fine as long as there's a one-to-one mapping between input and output, but that assumption breaks down pretty quickly once we move to real language tasks. This notebook looks at why that breaks down and how the Encoder-Decoder architecture fixes it.


## **Problems with a Vanilla RNN**

1. Variable length of input and output
2. Vanishing Gradient
3. Limited Context Window

We already looked at the vanishing gradient problem and limited context window in the RNN notebook (that's basically why [LSTM](https://www.geeksforgeeks.org/deep-learning/deep-learning-introduction-to-long-short-term-memory/) and [GRU](https://www.geeksforgeeks.org/machine-learning/gated-recurrent-unit-networks/) exist), so lets focus on the **variable length** problem here, since thats what motivates the Encoder-Decoder design.


### **Variable Length of Input and Output**

A Recurrent Neural Network, or RNN, is a network that operates on a sequence and uses its own output as input for subsequent steps.

In sequence prediction with a single RNN, every input corresponds to an output. This works well when there is a one-to-one alignment between inputs and outputs, such as:

- Part-of-speech tagging
- Named entity recognition
- Frame-by-frame video labeling

**Part-of-Speech (POS) Tagging:** Assigns a grammatical label (noun, verb, adjective, etc.) to each word in a sentence.

```
Input : The    cat    sat    on    the    mat
Output: DT     NN     VBD    IN    DT     NN
```

Where `DT` -> Determiner, `NN` -> Noun, `VBD` -> Verb (past tense), `IN` -> Preposition.

**Named Entity Recognition (NER):** Identifies and classifies named entities such as people, organizations and locations in a sentence.

```
Input : Steve  Jobs   founded  Apple   in   California
Output: B-PER  I-PER  O        B-ORG   O    B-LOC
```

Where `B-PER` -> beginning of a person's name, `I-PER` -> continuation of a person's name, `B-ORG` -> beginning of an organization, `B-LOC` -> beginning of a location, `O` -> not an entity.

> In both tasks, every input word corresponds to exactly one output label, which is why a simple RNN is enough here.


### **So where's the problem?**

The one-to-one assumption falls apart in tasks like:

- **Machine translation** - an English sentence may translate into a longer or shorter French sentence.
- **Text summarization** - a long document becomes a short summary.
- **Speech recognition** - many audio frames map to a much shorter sequence of words.
- **Image captioning** - a single image produces a sequence of words.

In these tasks, there isn't necessarily one output per input, the model usually needs to read the *entire* input before it can start generating anything, and the output length itself is variable.

Take Machine Translation as an example:

| # | Input Sequence | Output Sequence |
|---|----------------|-----------------|
| 1 | English: I am hungry. | Nepali: मलाई भोक लागेको छ। |
| 2 | English: I ate. | German: Ich habe gegessen. |
| 3 | English: I will become the Pirate King! | Japanese: 海賊王に俺はなる！ (*Kaizoku-o ni ore wa naru!*) |

Or cases where one language squeezes an entire idea into a single word while another needs a whole phrase to say the same thing:

| # | Input Sequence | Output Sequence |
|---|----------------|-----------------|
| 4 | German: Ubermorgen | English: the day after tomorrow |
| 5 | German: Vorgestern | English: the day before yesterday |
| 6 | Japanese: 木漏れ日 (*Komorebi*) | English: sunlight filtering through the leaves of trees |
| 7 | Japanese: 積読 (*Tsundoku*) | English: buying books and never reading them |

There's clearly no fixed word-to-word alignment we can rely on here, an RNN that emits one output per input token just cant express this. This is exactly the gap the **Encoder-Decoder** architecture is built to close.


## **The Encoder-Decoder Idea**

Instead of forcing an output at every timestep, we split the job into two RNNs:

- An **Encoder** that reads the whole input sequence and compresses it into a single vector (its final hidden state).
- A **Decoder** that takes that vector and generates the output sequence, one token at a time, until it decides to stop.

Since the encoder finishes reading the entire input before the decoder starts producing anything, the two sequences no longer need to have matching lengths, the model is free to output as many or as few words as it needs.

A Sequence to Sequence network, or seq2seq network, or Encoder-Decoder network is basically this pair of RNNs working together: the encoder reads an input sequence and outputs a single vector, and the decoder reads that vector to produce an output sequence.


## **Why not just feed a one-hot vector into the RNN?**

Before building the encoder, its worth pausing on a question that comes up naturally: every word can already be uniquely represented with a one-hot vector, so why bother with an embedding layer at all before feeding words into the RNN?

### What is a one-hot vector?

A one-hot vector represents a categorical value using a vector of all zeros except a single `1`. Say our vocabulary has five words:

| Index | Word |
|---|---|
| 0 | hello |
| 1 | world |
| 2 | I |
| 3 | love |
| 4 | AI |

```
hello -> [1, 0, 0, 0, 0]
world -> [0, 1, 0, 0, 0]
I     -> [0, 0, 1, 0, 0]
love  -> [0, 0, 0, 1, 0]
AI    -> [0, 0, 0, 0, 1]
```

The position of the `1` tells us which word it is. A vocabulary of 50,000 words needs a vector of length 50,000 for *every single word*.

### Can we feed this directly into an RNN?

Technically yes, an RNN just accepts vectors as input so there's no mathematical issue with feeding it a one-hot vector directly. But there are a few practical problems with doing this.

**Problem 1: One-hot vectors are huge.** A vocabulary of 100,000 words means every word becomes a 100,000-dimensional vector, and a sentence like *"I love machine learning"* would need the RNN to process four of these massive, almost entirely empty vectors.

**Problem 2: One-hot vectors carry no meaning.** All a one-hot vector tells the model is *"this is word number 523"*, nothing about what the word actually means. Consider:

```
king  -> [1,0,0,0]
queen -> [0,1,0,0]
car   -> [0,0,1,0]
truck -> [0,0,0,1]
```

From these vectors alone the model has no way to know that *king* is closer in meaning to *queen* than it is to *car*. The distance between `king` and `queen` is exactly the same as the distance between `king` and `banana`, because all one-hot vectors are mutually orthogonal.

**Problem 3: The RNN has to learn a huge transformation just to compress the input.** Say our vocabulary is 100,000 words and the RNN's hidden size is 256, the model needs a weight matrix of shape `100,000 x 256` (about 25.6 million parameters) purely to squeeze the one-hot input down into something usable. Thats a lot of parameters spent just getting the input into a workable shape.

### What does an embedding layer do instead?

An embedding layer swaps out the giant one-hot vector for a small, dense, *learned* vector:

```
word index -> Embedding layer -> 256-dimensional dense vector -> RNN
```

For example, after training we might get something like:

```
king:  [0.21, 0.52, 0.18, 0.76]
queen: [0.20, 0.49, 0.22, 0.71]
```

Now the vectors are actually informative, the model can learn that `king` and `queen` sit close to each other, and that `dog` and `cat` do too, because their embeddings end up nearby.

### Is an embedding just a faster one-hot lookup?

Yes, mathematically, `one-hot vector x embedding matrix` gives the exact same result as an embedding lookup. If our vocabulary is `{0: cat, 1: dog, 2: car}` and the embedding matrix is:

```
cat  [0.2, 0.5, 0.1]
dog  [0.3, 0.6, 0.2]
car  [0.8, 0.1, 0.7]
```

then the one-hot vector for `dog` (`[0,1,0]`) multiplied by this matrix simply selects row `[0.3, 0.6, 0.2]`. An embedding layer does this same selection directly, without ever materializing the large one-hot vector in the first place.

### Why are the embedding values trainable?

Embeddings usually start out random and get nudged during training so that words appearing in similar contexts move closer together. For instance, given the sentences:

```
The dog chased the ball.
The cat chased the mouse.
```

the model gradually learns `dog ~ cat`, purely because they behave similarly in these sentences.

### Could the RNN just learn everything from one-hot vectors directly?

In theory yes, but it would come at a cost: a huge input dimension, slower training, way more parameters, and the model having to rediscover word relationships completely from scratch instead of being handed a compact representation upfront. Splitting the problem into *(1) learn a meaningful dense representation of the word* and *(2) process that representation with the RNN* makes learning far more efficient, which is exactly what the embedding layer buys us.

### The full pipeline, put together

```text
Sentence: "I am Groot"
      |
      v
Word indices: [12, 45, 891]
      |
      v
Embedding layer: [[0.23,0.54,0.12], [0.11,0.72,0.43], [0.88,0.21,0.34]]
      |
      v
Encoder-Decoder
      |
      v
Prediction
```

So, a one-hot vector is useful because it uniquely identifies a word, but its a poor representation for actually *learning* language, its huge, sparse, and carries zero semantic information. An embedding layer gives us a compact, dense representation instead, which is why every implementation below starts with `nn.Embedding` rather than raw one-hot vectors.


## **Implementation**

Now that the theory's out of the way, lets actually build one. We'll train an Encoder-Decoder to translate French sentences into English, following the [PyTorch seq2seq tutorial](https://pytorch.org/tutorials/intermediate/seq2seq_translation_tutorial.html) fairly closely.

**Requirements**


In [1]:
from __future__ import unicode_literals, print_function, division
from io import open
import unicodedata
import re
import random

import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F

import numpy as np
from torch.utils.data import TensorDataset, DataLoader, RandomSampler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device = {device}")


Using device = cuda


**Note:** No `torch.set_default_device()` call here on purpose. Every tensor below already gets placed with `.to(device)` or `device=device` explicitly, so it isn'''t needed. Setting a global default device actually breaks things on a CUDA machine, `DataLoader`'''s `RandomSampler` creates an internal CPU generator for shuffling regardless of the default device, and once the default is `cuda` that mismatch throws `RuntimeError: Expected a '''cuda''' device type for generator but found '''cpu'''`.

### **Loading Data**

Download the dataset from here: [https://download.pytorch.org/tutorial/data.zip](https://download.pytorch.org/tutorial/data.zip)

Then extract it, we need `data/eng-fra.txt`. Its a tab-separated list of English-French translation pairs.

<p align="center">
  <img src="./images/DataPreparation.drawio.png" alt="Data Preparation">
</p>

To go from a raw sentence to something a model can use, we need a `word2index`, `index2word`, and a `word2count` mapping, so lets build a small class to hold those for each language.


In [19]:
SOS_token = 0 # Start of the Sentence
EOS_token = 1 # End of the Sentence

class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS"}
        self.n_words = 2  # Count SOS and EOS

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1


`addSentence` needs a normalized sentence to work with, so lets write the functions that read the file, split it into lines, build (input, target) pairs, and normalize the text (lowercasing, stripping accents, trimming punctuation, etc.).

In [20]:
# Turn a Unicode string to plain ASCII, thanks to
# https://stackoverflow.com/a/518232/2809427
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

# Lowercase, trim, and remove non-letter characters
def normalizeString(s):
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z!?]+", r" ", s)
    return s.strip()


In [21]:
def readLangs(path: str):
    lang1 = 'eng'; lang2 = 'fra'
    print("Reading lines...")

    # Read the file and split into lines
    lines = open(path, encoding='utf-8').\
        read().strip().split('\n')

    # Split every line into pairs and normalize (english to french)
    pairs = [[normalizeString(s) for s in l.split('\t')] for l in lines]

    # Reverse pairs: English-French -> French-English
    pairs = [list(reversed(p)) for p in pairs]

    # Input is French, output is English
    input_lang = Lang(lang2)
    output_lang = Lang(lang1)

    return input_lang, output_lang, pairs


There are a lot of sentence pairs in this dataset and we want to train fast, so lets trim it down to short, simple sentences only. We'll cap the length at 10 words (including the ending punctuation) and only keep sentences of the form *"I am ..."*, *"He is ..."*, etc. (accounting for the apostrophes we already normalized away). This keeps the vocabulary small and training quick.

In [22]:
MAX_LENGTH = 5

eng_prefixes = (
    "i am ", "i m ",
    "he is", "he s ",
    "she is", "she s ",
    "you are", "you re ",
    "we are", "we re ",
    "they are", "they re "
)

def filterPair(p):
    return len(p[0].split(' ')) < MAX_LENGTH and \
        len(p[1].split(' ')) < MAX_LENGTH and \
        p[1].startswith(eng_prefixes)


def filterPairs(pairs):
    return [pair for pair in pairs if filterPair(pair)]


Now lets bundle everything together into one function that reads the file, filters the pairs and builds the vocabulary for both languages.

In [23]:
def prepareData(path):
    input_lang, output_lang, pairs = readLangs(path)
    print("Read %s sentence pairs" % len(pairs))
    pairs = filterPairs(pairs)
    print("Trimmed to %s sentence pairs" % len(pairs))
    print("Counting words...")
    for pair in pairs:
        input_lang.addSentence(pair[0])
        output_lang.addSentence(pair[1])
    print("Counted words:")
    print(input_lang.name, input_lang.n_words)
    print(output_lang.name, output_lang.n_words)
    return input_lang, output_lang, pairs


In [24]:
PATH = r'data/eng-fra.txt'

input_lang, output_lang, pairs = prepareData(PATH)
print(random.choice(pairs))

output_lang.word2index['am']  # try different English words.


Reading lines...
Read 135842 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967
['je me sens bien', 'i m feeling good']


15

**Note:** We've only inserted lowercase words into `word2index`, so looking up an uppercase word will throw a `KeyError`.

## **Encoder-Decoder Network**

As discussed above, a Seq2Seq / Encoder-Decoder network is a pair of RNNs, an encoder that reads the input sequence and outputs a single vector, and a decoder that reads that vector to produce the output sequence.

### **Encoder**

The Encoder is an RNN that processes the input one word at a time. Since it doesn't need to produce an output at every step (unlike the decoder), we only care about the hidden state *after* it has read the entire input sequence, that final hidden state is what gets passed on. This matches what the [paper](https://emnlp2014.org/papers/pdf/EMNLP2014179.pdf) describes.


In [8]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, dropout_p=0.1):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(input_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, input):
        embedded = self.dropout(self.embedding(input))
        output, hidden = self.rnn(embedded)
        return output, hidden


Here we convert the input word indices into dense embedding vectors (exactly the reasoning we walked through above), during training these embeddings are learned and gradually start capturing the semantic relationships between words.

### **Decoder**

The Decoder is another RNN, it takes the encoder's output vector and generates a sequence of words. Specifically:

- We only use the *last* output/hidden state of the encoder, this is called the **context vector**, since it encodes context from the entire input sequence.
- This context vector becomes the *initial* hidden state of the decoder.
- The decoder's first input token is the start-of-string `<SOS>` token.


In [9]:
class DecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(DecoderRNN, self).__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)

    def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):
        batch_size = encoder_outputs.size(0)
        decoder_input = torch.empty(batch_size, 1, dtype=torch.long, device=device).fill_(SOS_token)
        decoder_hidden = encoder_hidden
        decoder_outputs = []

        for i in range(MAX_LENGTH):
            decoder_output, decoder_hidden  = self.forward_step(decoder_input, decoder_hidden)
            decoder_outputs.append(decoder_output)

            if target_tensor is not None:
                # Teacher forcing: Feed the target as the next input
                decoder_input = target_tensor[:, i].unsqueeze(1) # Teacher forcing
            else:
                # Without teacher forcing: use its own predictions as the next input
                _, topi = decoder_output.topk(1) # values, index of the highest-scoring word
                decoder_input = topi.squeeze(-1).detach()  # detach from history as input (removes the last dimension before detaching)

        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)
        return decoder_outputs, decoder_hidden, None # We return `None` for consistency in the training loop

    def forward_step(self, input, hidden):
        output = self.embedding(input)
        output = F.relu(output)
        output, hidden = self.rnn(output, hidden)
        output = self.out(output)
        return output, hidden


**Without teacher forcing / during inference:** we need the word index rather than the raw tensor, because the embedding layer expects an index, not a value.

**Why `detach()`?** PyTorch normally tracks every operation to build the computation graph for backpropagation.

```
decoder step 1 -> prediction -> decoder step 2 -> prediction
```

Without detaching, PyTorch would try to backpropagate through this entire generation chain, one prediction feeding the next. `detach()` tells it: treat this predicted token as a fresh input, don't drag along the previous computation history.

**`MAX_LENGTH`** here refers to the maximum length of the *generated* output sentence, not the input sentence.


**Encoder vs Decoder, side by side:**

| Aspect | Encoder RNN | Decoder RNN |
| --- | --- | --- |
| Purpose | Read and understand an input sequence | Generate an output sequence |
| Example | Input: "I love cats" | Output: "J'aime les chats" |
| Do we know the whole sequence beforehand? | Yes | No |
| Where is the loop? | Inside `nn.RNN` | In our `for i in range(MAX_LENGTH)` loop |
| Code pattern | `output, hidden = self.rnn(sequence)` | `for i in range(MAX_LENGTH): output, hidden = self.rnn(one_token, hidden)` |
| Number of RNN calls | Usually 1 call | Usually `MAX_LENGTH` calls |
| Input to RNN | Entire sequence | One token at a time |
| Sequence length given to RNN | `seq_len = input sentence length` | `seq_len = 1` per call |
| Hidden state recurrence | Automatic | Automatic |
| Output recurrence | Usually none | Manually created |

**Code comparison:**

| Component | Encoder | Decoder |
| --- | --- | --- |
| Calling the recurrent layer | `output, hidden = self.rnn(embedded)` | `output, hidden = self.rnn(input, hidden)` |
| Number of calls | One call processes the whole sequence | One call per generated token |
| Loop location | Inside PyTorch | Inside our `for i in range(MAX_LENGTH)` loop |
| Recurrent unit sees | All input tokens | One generated token at each step |
| Generates tokens? | No | Yes |


## **Training**

### **Preparing the Training Data**

For each pair, we need an input tensor (word indices for the input sentence) and a target tensor (word indices for the target sentence). While building these we append the `EOS` token to both sequences.


In [10]:
def indexesFromSentence(lang, sentence):
    return [lang.word2index[word] for word in sentence.split(' ')]

def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(1, -1)

def tensorsFromPair(pair):
    input_tensor = tensorFromSentence(input_lang, pair[0])
    target_tensor = tensorFromSentence(output_lang, pair[1])
    return (input_tensor, target_tensor)

def get_dataloader(batch_size):
    input_lang, output_lang, pairs = prepareData(path=PATH)

    n = len(pairs)
    input_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)
    target_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)

    for idx, (inp, tgt) in enumerate(pairs):
        inp_ids = indexesFromSentence(input_lang, inp)
        tgt_ids = indexesFromSentence(output_lang, tgt)
        inp_ids.append(EOS_token)
        tgt_ids.append(EOS_token)
        input_ids[idx, :len(inp_ids)] = inp_ids
        target_ids[idx, :len(tgt_ids)] = tgt_ids

    train_data = TensorDataset(torch.LongTensor(input_ids).to(device),
                               torch.LongTensor(target_ids).to(device))

    train_sampler = RandomSampler(train_data)
    train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)
    return input_lang, output_lang, train_dataloader


### **Training Loop**

In [11]:
def train_epoch(dataloader, encoder, decoder, encoder_optimizer,
          decoder_optimizer, criterion):

    total_loss = 0
    for data in dataloader:
        input_tensor, target_tensor = data

        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, _, _ = decoder(encoder_outputs, encoder_hidden, target_tensor) # using teacher forcing

        loss = criterion(
            decoder_outputs.view(-1, decoder_outputs.size(-1)),
            target_tensor.view(-1)
        )
        loss.backward()

        encoder_optimizer.step()
        decoder_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)


In [12]:
import time
import math

def asMinutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)

def timeSince(since, percent):
    now = time.time()
    s = now - since
    es = s / (percent)
    rs = es - s
    return '%s (- %s)' % (asMinutes(s), asMinutes(rs))


In [13]:
import matplotlib.pyplot as plt
plt.switch_backend('agg')
import matplotlib.ticker as ticker
import numpy as np

def showPlot(points):
    plt.figure()
    fig, ax = plt.subplots()
    # this locator puts ticks at regular intervals
    loc = ticker.MultipleLocator(base=0.2)
    ax.yaxis.set_major_locator(loc)
    plt.plot(points)


In [14]:
def train(train_dataloader, encoder, decoder, n_epochs, learning_rate=0.001,
               print_every=100, plot_every=100):
    start = time.time()
    plot_losses = []
    print_loss_total = 0  # Reset every print_every
    plot_loss_total = 0  # Reset every plot_every

    encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)
    criterion = nn.NLLLoss()

    for epoch in range(1, n_epochs + 1):
        loss = train_epoch(train_dataloader, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion)
        print_loss_total += loss
        plot_loss_total += loss

        if epoch % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start, epoch / n_epochs),
                                        epoch, epoch / n_epochs * 100, print_loss_avg))

        if epoch % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            plot_losses.append(plot_loss_avg)
            plot_loss_total = 0

    showPlot(plot_losses)


### **Evaluation Code**

In [15]:
def evaluate(encoder, decoder, sentence, input_lang, output_lang):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, decoder_hidden, decoder_attn = decoder(encoder_outputs, encoder_hidden)

        _, topi = decoder_outputs.topk(1)
        decoded_ids = topi.squeeze()

        decoded_words = []
        for idx in decoded_ids:
            if idx.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            decoded_words.append(output_lang.index2word[idx.item()])
    return decoded_words, decoder_attn


In [16]:
def evaluateRandomly(encoder, decoder, n=10):
    for i in range(n):
        pair = random.choice(pairs)
        print('>', pair[0])
        print('=', pair[1])
        output_words, _ = evaluate(encoder, decoder, pair[0], input_lang, output_lang)
        output_sentence = ' '.join(output_words)
        print('<', output_sentence)
        print('')


### **Training and Evaluating**

In [17]:
hidden_size = 128
batch_size = 32
EPOCHS = 200

input_lang, output_lang, train_dataloader = get_dataloader(batch_size)

encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder = DecoderRNN(hidden_size, output_lang.n_words).to(device)

train(train_dataloader, encoder, decoder, EPOCHS, print_every=5, plot_every=5)


Reading lines...
Read 135842 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967
0m 8s (- 5m 31s) (5 2%) 1.9765
0m 14s (- 4m 39s) (10 5%) 1.3018
0m 21s (- 4m 30s) (15 7%) 1.1086
0m 29s (- 4m 21s) (20 10%) 0.9841
0m 34s (- 4m 4s) (25 12%) 0.8785
0m 40s (- 3m 50s) (30 15%) 0.7796
0m 46s (- 3m 38s) (35 17%) 0.6892
0m 52s (- 3m 29s) (40 20%) 0.6083
0m 58s (- 3m 20s) (45 22%) 0.5330
1m 3s (- 3m 11s) (50 25%) 0.4663
1m 9s (- 3m 3s) (55 27%) 0.4044
1m 15s (- 2m 56s) (60 30%) 0.3526
1m 21s (- 2m 49s) (65 32%) 0.3077
1m 27s (- 2m 42s) (70 35%) 0.2680
1m 33s (- 2m 35s) (75 37%) 0.2365
1m 38s (- 2m 28s) (80 40%) 0.2060
1m 44s (- 2m 21s) (85 42%) 0.1812
1m 52s (- 2m 17s) (90 45%) 0.1645
1m 59s (- 2m 11s) (95 47%) 0.1439
2m 5s (- 2m 5s) (100 50%) 0.1312
2m 11s (- 1m 59s) (105 52%) 0.1192
2m 17s (- 1m 52s) (110 55%) 0.1081
2m 23s (- 1m 46s) (115 57%) 0.1042
2m 29s (- 1m 39s) (120 60%) 0.0907
2m 36s (- 1m 33s) (125 62%) 0.0880
2m 42s (- 1m 27s) (130 65%) 0.

In [18]:
encoder.eval()
decoder.eval()
evaluateRandomly(encoder, decoder)


> nous avons toutes peur
= we re all scared
< we re all scared <EOS>

> vous etes beaux
= you re beautiful
< you are my father <EOS>

> vous analysez trop
= you re over analyzing
< we re the problem <EOS>

> j y retourne
= i m going back
< he s tickled pink <EOS>

> tu es tres emotive
= you re very emotional
< you re very emotional <EOS>

> je suis gave
= i m stuffed
< i m going downtown <EOS>

> je suis le capitaine
= i m the captain
< i m the captain <EOS>

> je suis fatigue !
= i m tired
< i am tired <EOS>

> tu entends des choses
= you are hearing things
< you are hearing things <EOS>

> nous avons tous faim
= we re all hungry
< we re all hungry <EOS>



## **Discussion:**
The whole point of the Encoder-Decoder was to get around the one-to-one limitation of a vanilla RNN. The encoder reads the full French sentence and compresses it into a single context vector (its last hidden state), and the decoder unrolls that vector back into an English sentence one word at a time. Since the decoder isn't tied to the input length at all, it can produce a sentence thats longer, shorter, or the same length as the input, exactly the flexibility a plain RNN didn't have.

#### **What is teacher forcing doing here?**
During training, `decoder_input = target_tensor[:, i].unsqueeze(1)` means we feed the *actual* correct word as the next input instead of whatever the decoder just predicted. This is teacher forcing. It helps the model converge faster because a single wrong prediction early on doesn't cascade into every following step being wrong too. The tradeoff is that during evaluation the model never gets to see the ground truth, it only sees its own predictions (`evaluate()` calls the decoder with `target_tensor=None`), so there can be a gap between how good training loss looks and how good the actual generated sentences are.

#### **Why did we filter down to short "I am / he is" style sentences?**
Same reasoning as choosing `hidden_size` in the RNN notebook, the model needs to be matched to the dataset. A `hidden_size` of 128 and a `MAX_LENGTH` of 5 is only enough capacity to reliably learn a small, repetitive slice of sentence patterns. If we'd used the full 135,842 pairs with much longer sentences, this same architecture would struggle, because the entire sentence has to be squeezed into one fixed-size context vector no matter how long the input is. Trimming to short, similarly structured sentences (`eng_prefixes`) keeps the vocabulary small and the patterns consistent enough for a small model to actually learn them.

#### **Where does this architecture fall short?**
The single context vector is the bottleneck. For a 4-5 word sentence its fine, but as the input sentence gets longer, the encoder has to squash more and more information into that same fixed-size vector, and the decoder starts losing access to detail from earlier in the sentence. Thats the exact problem [attention](https://pytorch.org/tutorials/intermediate/seq2seq_translation_tutorial.html#attention-decoder) was introduced to fix, instead of relying on one context vector, the decoder gets to look back at *all* of the encoder's hidden states at every generation step.


## **Conclusion**
The Encoder-Decoder architecture was implemented using two RNNs, one to read and compress a French sentence into a context vector, and one to generate the equivalent English sentence from that vector. Training used teacher forcing on a filtered slice of the eng-fra dataset (short, "I am / he is" style sentences) with a hidden size of 128 over 200 epochs.

Looking at the sample outputs from `evaluateRandomly`, the model gets a good chunk of translations right or close to right, but still mixes up some words, especially on sentences it hasn't seen as many similar examples of. This lines up with what we'd expect, the dataset was filtered to keep training fast rather than exhaustive, and a single fixed-size context vector limits how much the model can hold onto for longer or less common sentence patterns.

This shows that unlike the vanilla RNN, the Encoder-Decoder setup can handle input and output sequences of different lengths without needing a one-to-one alignment, which is exactly what tasks like translation need. The next logical improvement here would be adding an attention mechanism so the decoder isn't limited to one compressed context vector and can instead refer back to the full input sequence while generating each word.
